In [ ]:
import pandas as pd
from collections import Counter
import re
import matplotlib.pyplot as plt
from google.colab import files
import spacy
import nltk

!python -m spacy download it_core_news_sm
nlp = spacy.load("it_core_news_sm")

# stopwords
nltk.download("stopwords")
from nltk.corpus import stopwords

stopwords_it = set(stopwords.words("italian"))

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = ""

In [ ]:
import pandas as pd
post = pd.read_csv("")

In [ ]:
!pip install transformers torch pandas
from transformers import pipeline
# cleaning the text
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

post["comment_message_clean"] = post["comment_message"].apply(clean_text)

# initialize an Italian model from Hugging Face.
sentiment_clf = pipeline("text-classification", model="MilaNLProc/feel-it-italian-sentiment")

# analysing the sentiment
def get_sentiment(text):
    if not text.strip():
        return None, None
    try:
        result = sentiment_clf(text[:512])
        return result[0]["label"], result[0]["score"]
    except:
        return None, None


post["sentiment_label"], post["sentiment_score"] = zip(*post["comment_message_clean"].apply(get_sentiment))

post.to_csv("post_con_sentiment.csv", index=False)


print(post.head())

In [ ]:
# Emotion analysis
emotion_clf = pipeline("text-classification", model="MilaNLProc/feel-it-italian-emotion")

def get_emotion(text):
    if not text.strip():
        return None, None
    try:
        result = emotion_clf(text[:512])
        return result[0]["label"], result[0]["score"]
    except:
        return None, None

post["emotion_label"], post["emotion_score"] = zip(*post["comment_message_clean"].apply(get_emotion))


post.to_csv("post_con_sentiment_emotion.csv", index=False)


print(post.head())

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'post' is your DataFrame
sentiment_counts = post["sentiment_label"].value_counts()

plt.figure(figsize=(6, 4))
sentiment_counts.plot(kind="bar", color=["skyblue", "salmon", "lightgreen"])

plt.title("Sentiment distribution in comments")
plt.xlabel("Sentiment")
plt.ylabel("Number of comments")
plt.ylim(0, 3000)
plt.xticks(rotation=0)

# --- SAVE TO PDF ---
# Using bbox_inches='tight' ensures the axis labels aren't cut off
plt.savefig("sentiment_distribution_1.pdf", format="pdf", bbox_inches='tight')

plt.show()

In [ ]:
# Assuming 'post' is your DataFrame
emotion_counts = post["emotion_label"].value_counts()

plt.figure(figsize=(8, 5)) # Slightly wider to accommodate emotion labels
emotion_counts.plot(kind="bar", color=["skyblue", "salmon", "lightgreen", "gold", "plum"]) # Added more colors just in case

plt.title("Emotion distribution in comments")
plt.xlabel("Emotion")
plt.ylabel("Number of comments")
plt.xticks(rotation=45) # Rotates labels to prevent overlapping

# --- SAVE TO PDF ---
# bbox_inches='tight' is crucial here to keep the rotated labels inside the file
plt.savefig("emotion_distribution_1.pdf", format="pdf", bbox_inches='tight')

plt.show()